# 01 - Bronze: land raw CSVs as files (no tables)

The original design (one Delta table per league per season) hit ~470
tables and triggered a Unity Catalog table-quota warning before Silver or
Gold even existed - see docs/schema_notes.md for the full writeup and
`00_reset_bronze_schemas.ipynb` for the cleanup of the old tables.

New design: Bronze is just a frozen file copy, no Delta tables, no Unity
Catalog quota usage at all. Real tables start at Silver, one per league
(not per season) - see `02_silver/01_england_premier_league.ipynb`.

Resumable: skips the copy entirely if the destination already exists.

In [ ]:
import os
import shutil

RAW_PATH = "/Volumes/workspace/default/football_raw"
BRONZE_PATH = "/Volumes/workspace/default/football_bronze"

In [ ]:
# Unity Catalog Volumes are catalog-registered objects - shutil/os can't
# create the volume itself (only files/dirs inside an existing one), so it
# must be created explicitly before copying anything into it
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.football_bronze")

if len(os.listdir(BRONZE_PATH)) > 0:
    print(f"{BRONZE_PATH} already has content - skipping copy (empty it first to force a re-copy)")
else:
    shutil.copytree(RAW_PATH, BRONZE_PATH, dirs_exist_ok=True)
    print(f"Copied {RAW_PATH} -> {BRONZE_PATH}")

## Verify - file counts should match the raw folder exactly

In [ ]:
def count_csvs(base):
    return sum(1 for _, _, files in os.walk(base) for f in files if f.endswith(".csv"))

raw_main = count_csvs(f"{RAW_PATH}/main_leagues")
raw_extra = count_csvs(f"{RAW_PATH}/extra_leagues")
bronze_main = count_csvs(f"{BRONZE_PATH}/main_leagues")
bronze_extra = count_csvs(f"{BRONZE_PATH}/extra_leagues")

print(f"main leagues:  raw={raw_main}  bronze={bronze_main}")
print(f"extra leagues: raw={raw_extra}  bronze={bronze_extra}")
assert raw_main == bronze_main and raw_extra == bronze_extra, "Bronze copy is incomplete"